# Aplicaciones con hdfstream + swiftsimio: visualización ligera de snapshots cosmológicos

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/gw-lvk-python-course/blob/main/Articulos/Aplicaciones_hdfstream_swiftsimio.ipynb)

Este notebook demuestra cómo combinar:

- **streaming** de datos HDF5 remotos con `hdfstream`,
- lectura y visualización de snapshots **SWIFT** con `swiftsimio`.

El objetivo es generar gráficos científicos interesantes sin descargar datasets gigantes, ideal para Google Colab.

## 0. Preparación del entorno

In [ ]:
import importlib, subprocess, sys

required = ["numpy", "matplotlib", "h5py", "hdfstream", "swiftsimio", "unyt"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print('Instalando:', ', '.join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print('Dependencias listas ✅')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import unyt as u
from pathlib import Path

from swiftsimio import Writer, load
from swiftsimio.objects import cosmo_array
from swiftsimio.metadata.writer.unit_systems import cosmo_units
from swiftsimio.visualisation import project_gas

import hdfstream

plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(42)

BOX_SIZE = 10.0  # Mpc
SCALE_FACTOR = 1.0
N_GAS = 12000
N_DM = 12000

## 1. Snapshot SWIFT sintético (ligero)

Creamos un mini-snapshot con gas y materia oscura usando `swiftsimio.Writer`.
Esto permite practicar visualización sin descargar archivos masivos.

In [ ]:
snapshot_path = Path("swift_mini_demo.hdf5")

if not snapshot_path.exists():
    boxsize = BOX_SIZE
    box = cosmo_array([boxsize] * 3, u.Mpc, comoving=True, scale_factor=SCALE_FACTOR, scale_exponent=1)
    w = Writer(unit_system=cosmo_units, boxsize=box, scale_factor=SCALE_FACTOR)

    gas_coords = rng.normal(loc=boxsize / 2, scale=boxsize * 0.15, size=(N_GAS, 3))
    gas_coords = np.clip(gas_coords, 0, boxsize)
    gas_coords = cosmo_array(gas_coords, u.Mpc, comoving=True, scale_factor=SCALE_FACTOR, scale_exponent=1)
    gas_vel = cosmo_array(rng.normal(0, 80, size=(N_GAS, 3)), u.km / u.s, comoving=True, scale_factor=SCALE_FACTOR, scale_exponent=1)

    r = np.linalg.norm(gas_coords.to_value(u.Mpc) - boxsize / 2, axis=1)
    temperature = 1e4 + 8e5 * np.exp(-r / (0.3 * boxsize))
    internal_energy = cosmo_array(
        temperature / 1e6, u.kb * u.K / u.solMass, comoving=True, scale_factor=SCALE_FACTOR, scale_exponent=-2
    )

    w.gas.coordinates = gas_coords
    w.gas.velocities = gas_vel
    w.gas.masses = cosmo_array(
        np.ones(N_GAS) * 1e6, u.solMass, comoving=True, scale_factor=SCALE_FACTOR, scale_exponent=0
    )
    w.gas.internal_energy = internal_energy
    w.gas.generate_smoothing_lengths()

    dm_coords = cosmo_array(
        rng.random((N_DM, 3)) * boxsize, u.Mpc, comoving=True, scale_factor=SCALE_FACTOR, scale_exponent=1
    )
    dm_vel = cosmo_array(
        rng.normal(0, 120, size=(N_DM, 3)), u.km / u.s, comoving=True, scale_factor=SCALE_FACTOR, scale_exponent=1
    )
    w.dark_matter.coordinates = dm_coords
    w.dark_matter.velocities = dm_vel
    w.dark_matter.masses = cosmo_array(
        np.ones(N_DM) * 5e6, u.solMass, comoving=True, scale_factor=SCALE_FACTOR, scale_exponent=0
    )

    w.write(snapshot_path)

print(f"Snapshot listo: {snapshot_path}")

In [ ]:
data = load(snapshot_path)

image = project_gas(data, resolution=256, project="masses")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im = axes[0].imshow(
    np.log10(image.to_value(u.solMass / u.Mpc**2) + 1e-6),
    origin="lower",
    cmap="magma",
)
axes[0].set_title("Superficie de masa de gas (log)")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
fig.colorbar(im, ax=axes[0], label="log10(Msun/Mpc^2)")

coords = data.gas.coordinates.to_value(u.Mpc)
center = BOX_SIZE / 2
radius = np.linalg.norm(coords - center, axis=1)

masses = data.gas.masses.to_value(u.solMass)
bins = np.linspace(0, BOX_SIZE / 2, 20)
mass_shell, edges = np.histogram(radius, bins=bins, weights=masses)
area = np.pi * (edges[1:] ** 2 - edges[:-1] ** 2)
surface_density = mass_shell / area

axes[1].plot(0.5 * (edges[1:] + edges[:-1]), surface_density)
axes[1].set_xlabel("Radio [Mpc]")
axes[1].set_ylabel("Σ_gas [Msun/Mpc^2]")
axes[1].set_title("Perfil radial de superficie de masa")

plt.tight_layout()
plt.show()

## 2. Streaming remoto con hdfstream

hdfstream permite leer porciones de un HDF5 remoto sin descargarlo completo.
Usaremos un snapshot público de EAGLE (si el servidor está disponible).

In [ ]:
server = "cosma"
remote_path = "/EAGLE/Fiducial_models/RefL0012N0188/snapshot_028_z000p000/snap_028_z000p000.0.hdf5"

try:
    snap = hdfstream.open(server, remote_path)

    sample_n = 25000
    dm_coords = snap["PartType1/Coordinates"][:sample_n]
    gas_temp = snap["PartType0/Temperature"][:sample_n]
    gas_density = snap["PartType0/Density"][:sample_n]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist2d(dm_coords[:, 0], dm_coords[:, 1], bins=120, cmap="viridis")
    axes[0].set_title("Distribución DM (hdfstream)")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("y")

    axes[1].scatter(
        np.log10(gas_density + 1e-30),
        np.log10(gas_temp + 1e-30),
        s=4,
        alpha=0.35,
        color="tab:orange",
    )
    axes[1].set_xlabel("log10(Densidad)")
    axes[1].set_ylabel("log10(Temperatura)")
    axes[1].set_title("Diagrama fase del gas (hdfstream)")

    plt.tight_layout()
    plt.show()

except Exception as exc:
    print("No se pudo conectar al servidor hdfstream:", exc)
    print("Sugerencia: si estás en Colab, revisa tu conexión o cambia la URL del servidor.")